# Parte Específica — EDA do Pump it Up

Projeto Final AVD 2026.2 — Grupo 6. Responsável: **Gustavo**.

Dataset: [Pump it Up: Data Mining the Water Table (DrivenData)](https://www.drivendata.org/competitions/7/pump-it-up-data-mining-the-water-table/). Classe: `status_group` (`functional`, `functional needs repair`, `non functional`).

> **Sobre o uso de IA neste notebook:** as células de código marcadas com `# [IA]` foram geradas com apoio de IA para *tornar a análise possível* (cálculos, tabelas e gráficos). Toda interpretação, discussão e decisão de tratamento é autoral do grupo e fica nas células "Interpretação autoral".

## 0. Pergunta de pesquisa e público-alvo

**Pergunta (versão inicial):** *Can you predict which water pumps are faulty?*

**Público-alvo:** gestor público (ex.: Ministério da Água da Tanzânia).

> **[Interpretação autoral — preencher]**

## Configuração

In [ ]:
# [IA] Claude Code – configuração (imports, caminhos, cores da classe e função para salvar figuras)
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Sobe pastas até achar a raiz do repositório (onde está o requirements.txt),
# para o notebook funcionar não importa de onde o Jupyter foi aberto.
ROOT = Path.cwd()
while not (ROOT / "requirements.txt").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data" / "parte_especifica"
OUT_DIR = ROOT / "out" / "parte_especifica"
OUT_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# Ordem e cores fixas da classe, para todos os gráficos da EDA lerem igual.
# Paleta Okabe-Ito (acessível para daltonismo, Aula 05). Pode trocar à vontade.
STATUS_ORDEM = ["functional", "functional needs repair", "non functional"]
STATUS_CORES = {
    "functional": "#0072B2",
    "functional needs repair": "#E69F00",
    "non functional": "#D55E00",
}


def salvar(fig, nome):
    """Salva a figura em out/parte_especifica/<nome>.png."""
    fig.savefig(OUT_DIR / f"{nome}.png", dpi=150, bbox_inches="tight")


print("Raiz do projeto:", ROOT)

## Carga dos dados

In [ ]:
# [IA] Claude Code – leitura do dataset bruto (df_raw nunca é alterado)
df_raw = pd.read_csv(DATA_DIR / "raw" / "pump_it_up_train.csv")
df = df_raw.copy()
print(f"{df.shape[0]:,} linhas x {df.shape[1]} colunas")
df.head()

## 1. Análise Estrutural

Tipos, dimensões, cardinalidade, duplicados, colunas constantes, colunas redundantes e consistência dos textos.

In [ ]:
# [IA] Claude Code – visão geral de cada coluna (tipo, nº de valores únicos, nulos e um exemplo)
resumo = pd.DataFrame({
    "tipo_pandas": df.dtypes.astype(str),
    "n_unicos": df.nunique(),
    "n_nulos": df.isna().sum(),
    "pct_nulos": (df.isna().mean() * 100).round(2),
    "exemplo": df.iloc[0],
})
resumo.sort_values("n_unicos")

In [ ]:
# [IA] Claude Code – separação das colunas em grupos usados no resto do notebook
COL_ID = "id"
COL_ALVO = "status_group"

# Medidas quantitativas
COLS_NUM = ["amount_tsh", "gps_height", "longitude", "latitude",
            "num_private", "population", "construction_year"]
# Numéricas no arquivo, mas são códigos de localização (na prática, qualitativas)
COLS_CODIGO = ["region_code", "district_code"]
COLS_DATA = ["date_recorded"]
# Todo o resto (exceto id e alvo) é qualitativo
COLS_CAT = [c for c in df.columns
            if c not in COLS_NUM + COLS_CODIGO + COLS_DATA + [COL_ID, COL_ALVO]]

print(f"Quantitativas ({len(COLS_NUM)}): {COLS_NUM}")
print(f"Códigos numéricos ({len(COLS_CODIGO)}): {COLS_CODIGO}")
print(f"Data ({len(COLS_DATA)}): {COLS_DATA}")
print(f"Qualitativas ({len(COLS_CAT)}): {COLS_CAT}")

In [ ]:
# [IA] Claude Code – linhas duplicadas, ids repetidos e colunas com um único valor
print("Linhas duplicadas (todas as colunas):", df.duplicated().sum())
print("Linhas duplicadas (ignorando o id):", df.drop(columns=COL_ID).duplicated().sum())
print("Ids repetidos:", df[COL_ID].duplicated().sum())

constantes = [c for c in df.columns if df[c].nunique(dropna=False) == 1]
print("Colunas com um único valor:", constantes)
for c in constantes:
    print(f"  {c}: {df[c].unique()}")

### 1.1 Colunas redundantes

Várias colunas aparecem em "famílias" (ex.: `extraction_type`, `extraction_type_group`, `extraction_type_class`). A tabela abaixo verifica, para cada par, se a coluna mais geral é só um agrupamento da mais detalhada: cada categoria da detalhada cai sempre em **uma única** categoria da geral.

In [ ]:
# [IA] Claude Code – verifica se as colunas "irmãs" são agrupamentos umas das outras
FAMILIAS = {
    "extraction_type": ["extraction_type", "extraction_type_group", "extraction_type_class"],
    "quantity": ["quantity", "quantity_group"],
    "payment": ["payment", "payment_type"],
    "water_quality": ["water_quality", "quality_group"],
    "source": ["source", "source_type", "source_class"],
    "waterpoint_type": ["waterpoint_type", "waterpoint_type_group"],
    "management": ["management", "management_group"],
}

linhas = []
for familia, cols in FAMILIAS.items():
    for detalhada, geral in zip(cols[:-1], cols[1:]):
        # maior nº de categorias "gerais" associadas a uma mesma categoria "detalhada"
        max_gerais = df.groupby(detalhada, dropna=False)[geral].nunique(dropna=False).max()
        linhas.append({
            "familia": familia,
            "detalhada": detalhada,
            "geral": geral,
            "n_cat_detalhada": df[detalhada].nunique(),
            "n_cat_geral": df[geral].nunique(),
            "geral_e_agrupamento_da_detalhada": max_gerais == 1,
        })
pd.DataFrame(linhas)

In [ ]:
# [IA] Claude Code – exemplos lado a lado: como as categorias se correspondem
display(pd.crosstab(df["quantity"], df["quantity_group"]))
display(pd.crosstab(df["payment"], df["payment_type"]))

### 1.2 Consistência entre `region` e `region_code`

In [ ]:
# [IA] Claude Code – quantos códigos existem por região e quantas regiões por código
codigos_por_regiao = df.groupby("region")["region_code"].unique().apply(sorted)
print("Regiões com mais de um region_code:")
display(codigos_por_regiao[codigos_por_regiao.apply(len) > 1])

regioes_por_codigo = df.groupby("region_code")["region"].unique().apply(sorted)
print("region_code com mais de uma região:")
display(regioes_por_codigo[regioes_por_codigo.apply(len) > 1])

### 1.3 Alta cardinalidade e grafias diferentes do mesmo nome

In [ ]:
# [IA] Claude Code – nº de categorias antes e depois de padronizar maiúsculas/espaços
linhas = []
for col in ["funder", "installer", "wpt_name", "subvillage", "scheme_name", "ward", "lga"]:
    s = df[col].dropna().astype(str)
    linhas.append({
        "coluna": col,
        "n_unicos": s.nunique(),
        "n_unicos_minusculo_sem_espacos": s.str.strip().str.lower().nunique(),
        "top_5": s.value_counts().head(5).to_dict(),
    })
pd.DataFrame(linhas)

> **[Interpretação autoral — preencher]**

## 2. Qualidade dos dados e mecanismo de ausência

In [ ]:
# TODO: faltantes explícitos e disfarçados, coocorrência dos zeros, ausência por região

> **[Interpretação autoral — preencher]**

## 3. Análise Distribucional

In [ ]:
# TODO: distribuição das numéricas, das principais categóricas e da classe

> **[Interpretação autoral — preencher]**

## 4. Análise Relacional (com `status_group`)

In [ ]:
# TODO: relação das variáveis com status_group, correlações e análise multivariada

> **[Interpretação autoral — preencher]**

## 5. Outliers

In [ ]:
# TODO: identificar outliers (amount_tsh, population etc.) e decidir o tratamento

> **[Interpretação autoral — preencher]**

## 6. Tratamento e exportação

Aplicar aqui **apenas** os tratamentos decididos e justificados nas seções 2 e 5, e exportar para `data/parte_especifica/clean/` (lido pelo dashboard).

Decisões pendentes (comentários da prof na planilha do checkpoint):
- [ ] Zeros de `population`, `construction_year`, `gps_height` e `longitude`/`latitude`: tratar como faltantes? Imputar como?
- [ ] `scheme_name` (48,5% ausente), `scheme_management`, `installer`, `funder`, `public_meeting`, `permit`: como tratar?
- [ ] `funder`/`installer` com `'0'` ou `'-'` e grafias diferentes do mesmo nome
- [ ] `num_private` (98,7% zero) e `recorded_by` (valor único): descartar?
- [ ] Colunas redundantes: qual versão de cada família manter?
- [ ] Alta cardinalidade (`wpt_name`, `subvillage`, `funder`, `installer`): manter, agrupar ou descartar?
- [ ] Outliers de `amount_tsh` e `population`
- [ ] Nomes de colunas sem acento/espaço (exigência do Streamlit)

In [ ]:
# TODO: tratamentos + df.to_csv(DATA_DIR / 'clean' / 'pump_it_up_clean.csv', index=False)

## 7. Síntese da EDA

> **[Interpretação autoral — preencher]**